In [1]:
from __future__ import annotations

In [2]:
%load_ext autoreload
%autoreload 3

In [3]:
import numpy as np
from typing import Any, Optional

from minitorch.tensor.tensor import Tensor
from minitorch.attention.attention import MultiHeadAttention
from minitorch.tokenization.tokenizer import CharTokenizer, BPETokenizer
from minitorch.dataloaders.dataloader import DataLoader
from minitorch.embendding.embed import EmbeddingLayer, Embedding, PositionalEncoding
from minitorch.losses.losses import MSE, SoftMaxCrossEntropy, BCEWithLogits, log_softmax
from minitorch.nn.layers import Linear, Module, LayerNormalization, Sequential, Residual, Dropout
from minitorch.activations.activations import Softmax, GELU, ReLU
from minitorch.optimizers.optim import AdamW
from minitorch.transformer.transformer import GPT
from minitorch.train.training import Trainer, CosineSchedule, clip_grad_norm

In [4]:
text_path = "C:\\Users\\User\\Downloads\\BIASHARA_Cleaned.txt"

with open(text_path, 'r', encoding='utf-8') as f:
    text = f.read()

In [5]:
#* split into training and val sets
n = 0.95
text_len = len(text)

chars = sorted(set(text))

context_length, batch_size, embed_dim = 8,8,32
#* tokenize the text
tokenizer = CharTokenizer() #* character level tokenization
tokenizer.build_vocab([text])
vocab_size = tokenizer.vocab_size

In [6]:
data = Tensor(tokenizer.encode(text))
train_data = data[:int(n*data.shape[0])]
val_data = data[int(n*data.shape[0]):]
train_loader = DataLoader(train_data, context_length)
val_loader = DataLoader(val_data, context_length)


In [20]:

###############################################################
#* define the hyperparameters
max_norm = 1
context_length, batch_size, embed_dim = 8,32,64
config = {
    'max_lr': 1,
    'min_lr': 0.00005,
    'embed_dim': embed_dim,
    'vocab_size': vocab_size,
    'max_seq_length': 1024,
    'n_heads': 8,
    'n_layers': 6,
    'dim': embed_dim,
    'dropout': 0.1,
    'max_iters': 100,
    'eval_iters': 1000,
    
}

#* ###########################################################
gpt = GPT(config)        #* model
optimizer = AdamW(gpt.parameters(), betas=(0.9, 0.999), lr=config['max_lr'],weight_decay=0.2)
scheduler = CosineSchedule(max_lr =config['max_lr'], min_lr=config['min_lr'], total_epochs= config['max_iters'])
trainer = Trainer(gpt, optimizer, max_norm, scheduler)

#* ###########################################################
#* training loop
print("Epoch          | Training Loss         | Validation Loss | ")
print("-" * 60)

for i in range(config['max_iters']):        
    training_loss = trainer.train_epoch(train_loader, accumulation_steps= config['eval_iters'])
    validation_loss = trainer.eval_epoch(val_loader)
    
    print(f'{i+1}             |   {training_loss:.4f}             | {validation_loss:.4f}')

Epoch          | Training Loss         | Validation Loss | 
------------------------------------------------------------
1             |   3.8578             | 3.4490
2             |   3.5788             | 3.9334
3             |   3.8448             | 3.5176
4             |   3.5858             | 3.2627
5             |   3.2525             | 3.3886
6             |   3.2796             | 3.1222
7             |   3.1445             | 3.4111
8             |   3.4476             | 3.1546
9             |   2.8977             | 3.3579
10             |   3.3431             | 3.7825
11             |   4.0064             | 3.0489
12             |   3.0020             | 3.1355
13             |   3.1038             | 3.1250
14             |   3.2439             | 2.9914
15             |   3.0783             | 3.1315
16             |   3.1407             | 2.8801
17             |   2.9294             | 3.0371
18             |   2.8956             | 2.9573
19             |   2.8751             | 2.

In [21]:
trainer.history

[autoreload of minitorch.optimizers.optim failed: Traceback (most recent call last):
  File "c:\Users\User\Desktop\babytorch\.venv\lib\site-packages\IPython\extensions\autoreload.py", line 274, in check
    superreload(m, reload, self.old_objects, self.shell)
  File "c:\Users\User\Desktop\babytorch\.venv\lib\site-packages\IPython\extensions\autoreload.py", line 475, in superreload
    module = reload(module)
  File "C:\Users\User\AppData\Roaming\uv\python\cpython-3.10.17-windows-x86_64-none\lib\importlib\__init__.py", line 169, in reload
    _bootstrap._exec(spec, module)
  File "<frozen importlib._bootstrap>", line 619, in _exec
  File "<frozen importlib._bootstrap_external>", line 883, in exec_module
  File "<frozen importlib._bootstrap>", line 241, in _call_with_frames_removed
  File "C:\Users\User\Desktop\babytorch\minitorch\optimizers\optim.py", line 37, in <module>
    class Optimizer:
  File "C:\Users\User\Desktop\babytorch\minitorch\optimizers\optim.py", line 45, in Optimizer
 

{'train_loss': [3.8578174114227295,
  3.5788381099700928,
  3.8448262214660645,
  3.5857787132263184,
  3.2525291442871094,
  3.279616355895996,
  3.1445016860961914,
  3.447627067565918,
  2.8976645469665527,
  3.3430702686309814,
  4.006369113922119,
  3.0019965171813965,
  3.1037585735321045,
  3.2439494132995605,
  3.078251600265503,
  3.1406612396240234,
  2.9294471740722656,
  2.8955588340759277,
  2.8750932216644287,
  3.014019012451172,
  2.9428651332855225,
  2.9437355995178223,
  2.8412301540374756,
  2.910860538482666,
  2.8561666011810303,
  3.2411272525787354,
  2.8357512950897217,
  2.9905447959899902,
  2.9243388175964355,
  2.8023526668548584,
  2.8669610023498535,
  2.9045143127441406,
  2.9673101902008057,
  2.8984222412109375,
  2.79897141456604,
  3.065070390701294,
  2.9410338401794434,
  2.942063808441162,
  3.162349224090576,
  2.8874285221099854,
  3.0281238555908203,
  2.9938528537750244,
  2.9863924980163574,
  2.93747615814209,
  2.9241349697113037,
  2.95176

In [8]:
total_params = 0.0

for param in gpt.parameters():
    if isinstance(param, list):
        for p in param:
            if isinstance(p, list):
                for sub_p in p:
                    total_params += np.prod(sub_p.shape)
            else:
                total_params += np.prod(p.shape)
    else:
        total_params += np.prod(param.shape)
        
total_params = total_params / 1e6  # Convert to millions
print(f"Total parameters: {total_params:.2f} M")

Total parameters: 0.36 M
